# CS224W — Electronics LightGCN Baseline
Run all cells in a fresh Google Colab runtime. Start with `USE_TOY_DATA = True` to check the pipeline without downloading data, then set it to False and rerun for the milestone experiment. The full Electronics review file must be downloaded even when sampling 1% of users. A GPU is helpful for the real run.

Both models use the same chronological split and rank the same warm-start catalog. This notebook reports validation metrics. The test split is reserved for final evaluation.

In [ ]:
%pip -q install duckdb huggingface_hub

In [ ]:
import math
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import torch
from huggingface_hub import hf_hub_download
from IPython.display import display
from torch import nn
from torch.nn import functional as F

USE_TOY_DATA = False  # Set True for a quick check without downloading reviews.
USER_SAMPLE_PCT = 1
SEED = 42
EVAL_CUTOFFS = (20, 50, 100)
EMBEDDING_DIM = 32
NUM_LAYERS = 2
EPOCHS = 5
STEPS_PER_EPOCH = 20
TRAIN_BATCH_SIZE = 8192
EVAL_BATCH_SIZE = 32

assert 1 <= USER_SAMPLE_PCT <= 100
torch.manual_seed(SEED)
rng = np.random.default_rng(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1. Build timestamped interactions and split chronologically

In [ ]:
workdir = Path("lightgcn_work")
workdir.mkdir(exist_ok=True)
con = duckdb.connect(str(workdir / "baseline.duckdb"))
con.execute("SET memory_limit='4GB'")
con.execute("SET threads=2")

if USE_TOY_DATA:
    # Toy timestamps create repeat users and warm items in validation.
    con.execute("""
        CREATE OR REPLACE TABLE interactions AS
        SELECT 'u' || u AS user_id,
               'i' || ((u + j) % 80) AS parent_asin,
               100 * j + u % 11 AS first_timestamp
        FROM range(120) AS users(u)
        CROSS JOIN range(10) AS offsets(j)
    """)
else:
    review_path = hf_hub_download(
        repo_id="McAuley-Lab/Amazon-Reviews-2023",
        repo_type="dataset",
        filename="raw/review_categories/Electronics.jsonl",
    )
    con.execute("""
        CREATE OR REPLACE TABLE interactions AS
        SELECT user_id, parent_asin, MIN(timestamp) AS first_timestamp
        FROM read_json(
            ?, format='newline_delimited',
            columns={
                'user_id': 'VARCHAR',
                'parent_asin': 'VARCHAR',
                'timestamp': 'BIGINT'
            }
        )
        WHERE user_id IS NOT NULL
          AND parent_asin IS NOT NULL
          AND timestamp IS NOT NULL
          AND md5_number_lower(user_id) % 100 < ?
        GROUP BY user_id, parent_asin
    """, [review_path, USER_SAMPLE_PCT])

val_start, test_start = con.execute("""
    SELECT quantile_disc(first_timestamp, 0.80),
           quantile_disc(first_timestamp, 0.90)
    FROM interactions
""").fetchone()

if val_start is None or val_start >= test_start:
    raise ValueError("Need more interactions with distinct timestamps.")

con.execute("""
    CREATE OR REPLACE TABLE baseline_split AS
    SELECT *, CASE WHEN first_timestamp < ? THEN 'train'
                   WHEN first_timestamp < ? THEN 'validation'
                   ELSE 'test' END AS split
    FROM interactions
""", [val_start, test_start])

display(con.execute("""
    SELECT split, COUNT(*) AS interactions,
           COUNT(DISTINCT user_id) AS users,
           COUNT(DISTINCT parent_asin) AS items
    FROM baseline_split
    GROUP BY split ORDER BY MIN(first_timestamp)
""").df())

## 2. Build the training graph
User and item IDs are assigned from the training split. Validation keeps only pairs whose endpoints are in training. The graph uses bidirectional edges with symmetric degree normalization.

In [ ]:
train = con.execute("""
    SELECT user_id, parent_asin
    FROM baseline_split WHERE split = 'train'
    ORDER BY user_id, parent_asin
""").df()
user_index, user_ids = pd.factorize(train["user_id"], sort=True)
item_index, item_ids = pd.factorize(train["parent_asin"], sort=True)
n_users, n_items = len(user_ids), len(item_ids)
if n_items < 2:
    raise ValueError("Need at least two training items.")

user_lookup = {value: idx for idx, value in enumerate(user_ids)}
item_lookup = {value: idx for idx, value in enumerate(item_ids)}
history = [set() for _ in range(n_users)]
for u, i in zip(user_index, item_index):
    history[u].add(i)
item_frequency = np.bincount(item_index, minlength=n_items)

item_nodes = item_index + n_users
src = np.concatenate([user_index, item_nodes])
dst = np.concatenate([item_nodes, user_index])
n_nodes = n_users + n_items
degree = np.bincount(src, minlength=n_nodes).astype(np.float32)
weights = 1 / np.sqrt(degree[src] * degree[dst])
adjacency = torch.sparse_coo_tensor(
    torch.from_numpy(np.stack([src, dst])),
    torch.from_numpy(weights),
    (n_nodes, n_nodes),
).coalesce().to(device)

validation = con.execute("""
    SELECT user_id, parent_asin
    FROM baseline_split WHERE split = 'validation'
    ORDER BY user_id, parent_asin
""").fetchall()
positives = {}
for raw_user, raw_item in validation:
    u, i = user_lookup.get(raw_user), item_lookup.get(raw_item)
    if u is not None and i is not None:
        positives.setdefault(u, set()).add(i)

warm_pairs = sum(map(len, positives.values()))
print(f"Warm-start validation: {warm_pairs:,}/{len(validation):,} pairs")
if not positives:
    raise ValueError("No warm-start validation pairs. Increase USER_SAMPLE_PCT.")

## 3. Evaluate Most Popular
For every validation user, rank training items after masking that user's training history. The sparse-item row uses the same ranked list at each cutoff and counts only validation positives with 1–5 training interactions. Metrics average across users with eligible positives in each group.
Recall and NDCG are reported at 20, 50, and 100. Each cutoff uses a prefix of the same ranked list.


In [ ]:
def evaluate(model_name, score_batch):
    results = {"overall": [], "sparse_items_1_to_5": []}
    max_k = max(EVAL_CUTOFFS)
    users = list(positives)
    for start in range(0, len(users), EVAL_BATCH_SIZE):
        batch = users[start:start + EVAL_BATCH_SIZE]
        scores = score_batch(batch)
        for row, u in enumerate(batch):
            scores[row, list(history[u])] = -torch.inf

        values, indices = scores.topk(min(max_k, n_items), dim=1)
        for u, ranked, valid in zip(
            batch, indices.tolist(), values.isfinite().tolist()
        ):
            relevant = positives[u]
            sparse = {i for i in relevant if item_frequency[i] <= 5}
            for group, targets in (("overall", relevant), ("sparse_items_1_to_5", sparse)):
                if targets:
                    ranks = [
                        rank for rank, (i, available) in enumerate(zip(ranked, valid))
                        if available and i in targets
                    ]
                    metrics = []
                    for k in EVAL_CUTOFFS:
                        hits = [rank for rank in ranks if rank < k]
                        dcg = sum(1 / math.log2(rank + 2) for rank in hits)
                        ideal = sum(
                            1 / math.log2(rank + 2)
                            for rank in range(min(k, len(targets)))
                        )
                        metrics.extend((len(hits) / len(targets), dcg / ideal))
                    results[group].append(metrics)

    rows = []
    for group, metrics in results.items():
        row = {"model": model_name, "group": group, "users": len(metrics)}
        means = np.mean(metrics, axis=0) if metrics else [np.nan] * (2 * len(EVAL_CUTOFFS))
        for index, k in enumerate(EVAL_CUTOFFS):
            row[f"Recall@{k}"] = means[2 * index]
            row[f"NDCG@{k}"] = means[2 * index + 1]
        rows.append(row)
    return pd.DataFrame(rows)


# Resolve ties by item ID while preserving the frequency ordering.
popular_scores = torch.as_tensor(item_frequency, dtype=torch.float64, device=device)
popular_scores -= torch.arange(n_items, device=device, dtype=torch.float64) / n_items
popular_results = evaluate(
    "Most Popular", lambda users: popular_scores.expand(len(users), -1).clone()
)
display(popular_results)

## 4. Train LightGCN with BPR
Each update samples training edges and unobserved items. LightGCN propagates over the full sparse training graph and averages its initial and propagated embeddings. The step count is fixed for this first run; an epoch here is a reporting interval of sampled updates.

In [ ]:
class LightGCN(nn.Module):
    def __init__(self):
        super().__init__()
        self.embedding = nn.Embedding(n_nodes, EMBEDDING_DIM)
        nn.init.normal_(self.embedding.weight, std=0.1)

    def forward(self):
        current = self.embedding.weight
        total = current
        for _ in range(NUM_LAYERS):
            current = torch.sparse.mm(adjacency, current)
            total = total + current
        return total / (NUM_LAYERS + 1)


# Users who saw the entire catalog have no eligible negative item.
eligible_users = np.array([len(items) < n_items for items in history])
eligible_edges = np.flatnonzero(eligible_users[user_index])
if not len(eligible_edges):
    raise ValueError("No training edges support negative sampling.")

model = LightGCN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
for epoch in range(EPOCHS):
    epoch_loss = 0.0
    for _ in range(STEPS_PER_EPOCH):
        sample = rng.choice(eligible_edges, size=min(TRAIN_BATCH_SIZE, len(eligible_edges)))
        users = user_index[sample]
        negative = rng.integers(n_items, size=len(sample))
        for j, u in enumerate(users):
            while negative[j] in history[u]:
                negative[j] = rng.integers(n_items)

        u = torch.as_tensor(users, device=device)
        p = torch.as_tensor(item_index[sample] + n_users, device=device)
        n = torch.as_tensor(negative + n_users, device=device)
        optimizer.zero_grad()
        embeddings = model()
        margins = (embeddings[u] * (embeddings[p] - embeddings[n])).sum(dim=1)
        loss = F.softplus(-margins).mean()
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    print(f"Epoch {epoch + 1}: BPR loss = {epoch_loss / STEPS_PER_EPOCH:.4f}")

## 5. Compare validation results

In [ ]:
model.eval()
with torch.no_grad():
    embeddings = model()
    user_embeddings = embeddings[:n_users]
    item_embeddings = embeddings[n_users:]
    lightgcn_results = evaluate(
        "LightGCN", lambda users: user_embeddings[users] @ item_embeddings.T
    )

results = pd.concat([popular_results, lightgcn_results], ignore_index=True)
results.insert(0, "dataset", "toy" if USE_TOY_DATA else "Electronics")
results["warm_validation_pairs"] = warm_pairs
results["total_validation_pairs"] = len(validation)
display(results)
output = workdir / ("toy_validation.csv" if USE_TOY_DATA else "electronics_validation.csv")
results.to_csv(output, index=False)
print("Saved:", output)